# 04 — Build the FULL tracked sequence dataset (IoU IDs + frozen group split)

**This is the replacement for the earlier pilot Notebook 04.** Use it only after Notebook 03 GPU extraction has finished and its coverage audit confirms all expected videos have complete NPZ outputs.

## Inputs
- Full tracked pose NPZs: `Notebook_2_Improved_v2/data/tracked_keypoints_iou_v3_full_v1/`
- Frozen train/validation/test manifest: `data/processed/sequences_group_split_v1/video_split_manifest.csv`
- Frozen class map: `data/processed/sequences_group_split_v1/label_map.json`

## Outputs
- A new timestamped dataset under `Notebook_2_Improved_v2/data/sequences_tracked_group_split_iou_v3_full_v1/`
- `X_train.npy`, `y_train.npy`, `videos_train.npy` and corresponding validation/test arrays
- Per-video and per-class reports plus `dataset_metadata.json`
- `data/CURRENT_TRACKED_SEQUENCE_DIR.txt`, which tells Notebook 05 which dataset to load

## Safety and correctness
- Reads Notebook 1 manifests and Notebook 2 NPZ files; it does not modify them.
- Reuses the existing recording-group assignments exactly; it never re-splits videos.
- Writes to a new timestamped folder. It does not overwrite the five-video pilot or prior sequence runs.
- Uses disk-backed `.npy` arrays and processes one NPZ at a time to limit RAM use.
- **Stops before sequence creation if any expected video is missing, unreadable, duplicated, or not marked as a complete full-video extraction.** Fix/resume Notebook 03 first in that case.
- Produces the same `[windows, 16, 51]` `X_*` format expected by Notebook 05. Notebook 05 derives velocity, acceleration and bone channels during training.

Do not run Notebook 05 using the old pilot sequence directory. Run this replacement Notebook 04 first.

In [1]:
# CELL 1 — Paths, settings, and frozen split validation
from pathlib import Path
from datetime import datetime
import json, gc, hashlib, os
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"
SOURCE_SEQ_DIR = PROJECT_DIR / "data" / "processed" / "sequences_group_split_v1"
SPLIT_PATH = SOURCE_SEQ_DIR / "video_split_manifest.csv"
LABEL_MAP_PATH = SOURCE_SEQ_DIR / "label_map.json"

# IMPORTANT: this is the FULL GPU/CPU-shared extraction folder, not the 5-video pilot.
TRACKED_DIR = NOTEBOOK2_DIR / "data" / "tracked_keypoints_iou_v3_full_v1"
OUT_ROOT = NOTEBOOK2_DIR / "data" / "sequences_tracked_group_split_iou_v3_full_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep consistent with the original group-aware dataset and Notebook 05.
SEQUENCE_LENGTH = 16
SEQUENCE_STRIDE = 4
MIN_VALID_FRAME_RATIO = 0.80
MIN_VALID_JOINTS = 8
EXPECTED_SOURCE_VIDEOS = 13319
EXPECTED_CLASSES = 101
EXPECTED_FRAME_STEP = 4

# A unique run folder prevents overwriting previous pilot or full sequence builds.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = OUT_ROOT / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)

assert SPLIT_PATH.is_file(), f"Frozen group split manifest not found: {SPLIT_PATH}"
assert LABEL_MAP_PATH.is_file(), f"Class map not found: {LABEL_MAP_PATH}"
assert TRACKED_DIR.is_dir(), f"Full tracked NPZ folder not found: {TRACKED_DIR}"

split_df = pd.read_csv(SPLIT_PATH)
label_map = json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))
label_to_id = {str(name): int(class_id) for class_id, name in label_map.items()}

required_cols = {"video_path", "label", "split", "video_id", "recording_group_key"}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f"Source manifest missing columns: {sorted(missing_cols)}"
assert split_df["video_path"].notna().all(), "Null video_path found in source split manifest."
assert split_df["recording_group_key"].notna().all(), "Null recording_group_key found."
assert set(split_df["split"].astype(str).unique()).issubset({"train", "val", "test"}), "Unexpected split name found."
assert len(split_df) == EXPECTED_SOURCE_VIDEOS, (
    f"Expected {EXPECTED_SOURCE_VIDEOS} source videos, but the frozen manifest has {len(split_df)}. "
    "Stop and verify the correct Notebook 1 split manifest before building."
)
assert len(label_map) == EXPECTED_CLASSES, f"Expected {EXPECTED_CLASSES} classes, found {len(label_map)}."

# Normalize Windows path strings for reliable manifest-to-NPZ matching.
def path_key(value):
    return os.path.normcase(os.path.normpath(str(value))).replace("/", "\\").casefold()

if split_df["video_path"].map(path_key).duplicated().any():
    duplicated_paths = split_df.loc[split_df["video_path"].map(path_key).duplicated(keep=False), "video_path"].head(10).tolist()
    raise ValueError(f"Duplicate source video paths in frozen manifest: {duplicated_paths}")

manifest_lookup = {}
for row in split_df.itertuples(index=False):
    manifest_lookup[path_key(row.video_path)] = {
        "video_path": str(row.video_path),
        "label": str(row.label),
        "split": str(row.split),
        "video_id": int(row.video_id),
        "recording_group_key": str(row.recording_group_key),
    }

# Confirm no recording group appears in more than one split.
group_sets = {
    s: set(split_df.loc[split_df["split"].astype(str).eq(s), "recording_group_key"].astype(str))
    for s in ["train", "val", "test"]
}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = group_sets[a] & group_sets[b]
    assert not overlap, f"Recording-group leakage in frozen manifest ({a}/{b}): {list(overlap)[:5]}"

print("Frozen source videos:", len(split_df))
print("Action classes:", len(label_map))
print("Original group split verified: zero overlap")
print("FULL tracked input folder:", TRACKED_DIR)
print("New sequence output folder:", OUT_DIR)
print("Window settings:", SEQUENCE_LENGTH, "frames | stride", SEQUENCE_STRIDE,
      "| minimum valid-frame ratio", MIN_VALID_FRAME_RATIO)

Frozen source videos: 13319
Action classes: 101
Original group split verified: zero overlap
FULL tracked input folder: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\tracked_keypoints_iou_v3_full_v1
New sequence output folder: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\sequences_tracked_group_split_iou_v3_full_v1\20261031_035536
Window settings: 16 frames | stride 4 | minimum valid-frame ratio 0.8


In [2]:
# CELL 2 — Full coverage audit and NPZ schema validation
# This cell does not create training sequences. It verifies the full extraction first.
tracked_files = sorted(
    p for p in TRACKED_DIR.rglob("*.npz")
    if ".tmp." not in p.name and ".partial." not in p.name
)
if not tracked_files:
    raise RuntimeError(f"No NPZ files found under {TRACKED_DIR}. Complete Notebook 03 first.")

required_npz_keys = {
    "keypoints", "boxes", "frame_indices", "has_person", "track_ids", "detection_scores",
    "all_keypoints", "all_boxes", "all_frame_indices", "all_track_ids", "all_detection_scores",
    "label", "video_path", "target_track_id", "fps", "total_frames", "frame_step",
    "person_selection", "complete_video", "max_decoded_frames", "decoded_frames",
    "max_people_per_frame", "inference_device"
}

tracked_rows, bad_rows = [], []
for i, npz_path in enumerate(tracked_files, 1):
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            absent = required_npz_keys - set(d.files)
            if absent:
                raise ValueError(f"Missing NPZ fields: {sorted(absent)}")

            keypoints = d["keypoints"]
            boxes = d["boxes"]
            frame_indices = d["frame_indices"]
            has_person = np.asarray(d["has_person"], dtype=bool)
            track_ids = np.asarray(d["track_ids"], dtype=np.int64)
            all_keypoints = d["all_keypoints"]
            all_track_ids = np.asarray(d["all_track_ids"], dtype=np.int64)

            video_path = str(d["video_path"].item())
            label = str(d["label"].item())
            complete_video = bool(d["complete_video"].item())
            decoded_frames = int(d["decoded_frames"].item())
            max_decoded_frames = int(d["max_decoded_frames"].item())
            frame_step = int(d["frame_step"].item())
            max_people = int(d["max_people_per_frame"].item())
            target_track_id = int(d["target_track_id"].item())
            inference_device = str(d["inference_device"].item())

            n = len(frame_indices)
            if keypoints.shape != (n, 17, 3):
                raise ValueError(f"keypoints shape {keypoints.shape}; expected {(n, 17, 3)}")
            if boxes.shape != (n, 4) or has_person.shape != (n,) or track_ids.shape != (n,):
                raise ValueError("Selected-target arrays have inconsistent dimensions")
            if d["detection_scores"].shape != (n,):
                raise ValueError("detection_scores length mismatch")
            if all_keypoints.ndim != 3 or all_keypoints.shape[1:] != (17, 3):
                raise ValueError("all_keypoints must have shape [observations, 17, 3]")
            if len(all_track_ids) != len(all_keypoints) or len(d["all_boxes"]) != len(all_keypoints):
                raise ValueError("All-person arrays have inconsistent lengths")
            if len(d["all_frame_indices"]) != len(all_keypoints) or len(d["all_detection_scores"]) != len(all_keypoints):
                raise ValueError("All-person metadata arrays have inconsistent lengths")
            if n == 0 or decoded_frames <= 0:
                raise ValueError("Empty or zero-frame extraction")

        key = path_key(video_path)
        meta = manifest_lookup.get(key)
        if meta is None:
            raise ValueError("video_path is not present in the frozen Notebook 1 split manifest")
        if label != meta["label"]:
            raise ValueError(f"Label mismatch: NPZ={label}, frozen manifest={meta['label']}")
        if label not in label_to_id:
            raise ValueError(f"Label not present in label_map.json: {label}")
        if frame_step != EXPECTED_FRAME_STEP:
            raise ValueError(f"frame_step={frame_step}; expected {EXPECTED_FRAME_STEP}")
        if max_people != 2:
            raise ValueError(f"max_people_per_frame={max_people}; expected 2")

        # The full extraction must reach EOF; pilot-limited or partial clips are rejected.
        if not complete_video:
            raise ValueError("complete_video is False")
        if max_decoded_frames != 0:
            raise ValueError(f"max_decoded_frames={max_decoded_frames}; expected 0 (full video)")

        selected_ids = np.unique(track_ids[has_person & (track_ids >= 0)])
        if len(selected_ids) > 1:
            raise ValueError(f"Selected pose sequence contains multiple track IDs: {selected_ids.tolist()}")
        if len(selected_ids) == 1 and target_track_id >= 0 and int(selected_ids[0]) != target_track_id:
            raise ValueError("Selected track IDs do not match target_track_id")

        tracked_rows.append({
            "npz_path": str(npz_path),
            "video_path": video_path,
            "path_key": key,
            "label": label,
            "split": meta["split"],
            "video_id": meta["video_id"],
            "recording_group_key": meta["recording_group_key"],
            "sampled_frames": int(n),
            "decoded_frames": int(decoded_frames),
            "frame_step": int(frame_step),
            "target_track_id": int(target_track_id),
            "target_pose_observations": int(has_person.sum()),
            "all_person_observations": int(len(all_track_ids)),
            "complete_video": True,
            "inference_device": inference_device,
        })
    except Exception as exc:
        bad_rows.append({"npz_path": str(npz_path), "error": repr(exc)})

    if i % 500 == 0 or i == len(tracked_files):
        print(f"Validated {i:,}/{len(tracked_files):,} NPZ files | valid={len(tracked_rows):,} | invalid={len(bad_rows):,}")

tracked_df = pd.DataFrame(tracked_rows)
expected_keys = set(manifest_lookup)
found_keys = set(tracked_df["path_key"]) if not tracked_df.empty else set()
missing_keys = expected_keys - found_keys
extra_or_missing_labels = len(bad_rows)

# A single source video must map to exactly one NPZ.
duplicate_rows = tracked_df[tracked_df["path_key"].duplicated(keep=False)] if not tracked_df.empty else pd.DataFrame()
duplicate_count = int(duplicate_rows["path_key"].nunique()) if not duplicate_rows.empty else 0

# Save audit files even when the full-coverage guard fails.
pd.DataFrame(bad_rows, columns=["npz_path", "error"]).to_csv(OUT_DIR / "unreadable_or_invalid_tracked_npz.csv", index=False)
pd.DataFrame({"video_path": [manifest_lookup[k]["video_path"] for k in sorted(missing_keys)]}).to_csv(
    OUT_DIR / "videos_missing_tracked_keypoints.csv", index=False
)
if not duplicate_rows.empty:
    duplicate_rows.drop(columns=["path_key"]).to_csv(OUT_DIR / "duplicate_source_video_npz.csv", index=False)

full_coverage = (
    len(found_keys) == len(expected_keys)
    and len(missing_keys) == 0
    and len(bad_rows) == 0
    and duplicate_count == 0
    and len(tracked_df) == len(expected_keys)
)

coverage_audit = {
    "expected_source_videos": int(len(expected_keys)),
    "npz_files_found": int(len(tracked_files)),
    "valid_unique_complete_videos": int(len(found_keys)),
    "missing_source_videos": int(len(missing_keys)),
    "invalid_or_incomplete_npz_files": int(len(bad_rows)),
    "duplicate_source_video_npz_groups": int(duplicate_count),
    "full_coverage": bool(full_coverage),
    "tracked_input_directory": str(TRACKED_DIR),
    "sequence_output_directory": str(OUT_DIR),
}
(OUT_DIR / "tracked_input_coverage_audit.json").write_text(json.dumps(coverage_audit, indent=2), encoding="utf-8")

print("\nFULL TRACKED INPUT COVERAGE AUDIT")
print(json.dumps(coverage_audit, indent=2))

if not full_coverage:
    raise RuntimeError(
        "Full tracked extraction coverage is not verified. No X/y training sequences were created. "
        "Review videos_missing_tracked_keypoints.csv and unreadable_or_invalid_tracked_npz.csv, "
        "finish or repair Notebook 03, then rerun this Notebook 04. Existing extraction files were not modified."
    )

# Keep exactly one row per source video and deterministic ordering.
tracked_df = tracked_df.sort_values(["split", "label", "video_id"]).reset_index(drop=True)
tracked_df.drop(columns=["path_key"]).to_csv(OUT_DIR / "tracked_video_manifest.csv", index=False)
print("\nFull coverage verified. Safe to build sequences.")

Validated 500/13,319 NPZ files | valid=500 | invalid=0
Validated 1,000/13,319 NPZ files | valid=1,000 | invalid=0
Validated 1,500/13,319 NPZ files | valid=1,500 | invalid=0
Validated 2,000/13,319 NPZ files | valid=2,000 | invalid=0
Validated 2,500/13,319 NPZ files | valid=2,500 | invalid=0
Validated 3,000/13,319 NPZ files | valid=3,000 | invalid=0
Validated 3,500/13,319 NPZ files | valid=3,500 | invalid=0
Validated 4,000/13,319 NPZ files | valid=4,000 | invalid=0
Validated 4,500/13,319 NPZ files | valid=4,500 | invalid=0
Validated 5,000/13,319 NPZ files | valid=5,000 | invalid=0
Validated 5,500/13,319 NPZ files | valid=5,500 | invalid=0
Validated 6,000/13,319 NPZ files | valid=6,000 | invalid=0
Validated 6,500/13,319 NPZ files | valid=6,500 | invalid=0
Validated 7,000/13,319 NPZ files | valid=7,000 | invalid=0
Validated 7,500/13,319 NPZ files | valid=7,500 | invalid=0
Validated 8,000/13,319 NPZ files | valid=8,000 | invalid=0
Validated 8,500/13,319 NPZ files | valid=8,500 | invalid=0
V

In [3]:
# CELL 3 — Normalize the selected person's skeleton and count windows (one NPZ at a time)
def normalize_pose(keypoints, boxes, target_valid):
    n = len(keypoints)
    out = np.zeros((n, 17, 3), dtype=np.float32)
    frame_valid = np.zeros(n, dtype=bool)

    for t in range(n):
        if not target_valid[t]:
            continue
        kp = np.asarray(keypoints[t], dtype=np.float32)
        box = np.asarray(boxes[t], dtype=np.float32)
        if not np.isfinite(box).all():
            continue
        x1, y1, x2, y2 = map(float, box)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue

        finite_xy = np.isfinite(kp[:, :2]).all(axis=1)
        if int(finite_xy.sum()) < MIN_VALID_JOINTS:
            continue

        # Joint positions are normalized relative to the selected person's bounding box.
        out[t, finite_xy, 0] = (kp[finite_xy, 0] - x1) / bw
        out[t, finite_xy, 1] = (kp[finite_xy, 1] - y1) / bh
        scores = np.nan_to_num(kp[:, 2], nan=0.0, posinf=0.0, neginf=0.0)
        out[t, :, 2] = np.clip(scores, 0.0, 1.0)
        out[t, ~finite_xy, :2] = 0.0
        out[t, ~finite_xy, 2] = 0.0
        frame_valid[t] = True

    return out, frame_valid


def iter_video_windows(npz_path):
    # Load only one video at a time; close the NPZ before yielding windows.
    with np.load(npz_path, allow_pickle=False) as d:
        keypoints = np.asarray(d["keypoints"], dtype=np.float32)
        boxes = np.asarray(d["boxes"], dtype=np.float32)
        frame_indices = np.asarray(d["frame_indices"], dtype=np.int32)
        has_person = np.asarray(d["has_person"], dtype=bool)
        track_ids = np.asarray(d["track_ids"], dtype=np.int64)
        target_track_id = int(d["target_track_id"].item())

    target_valid = has_person & (track_ids == target_track_id) & (target_track_id >= 0)
    poses, valid = normalize_pose(keypoints, boxes, target_valid)
    n = len(poses)
    if n >= SEQUENCE_LENGTH:
        starts = list(range(0, n - SEQUENCE_LENGTH + 1, SEQUENCE_STRIDE))
        final_start = n - SEQUENCE_LENGTH
        if starts and starts[-1] != final_start:
            starts.append(final_start)
        for start in starts:
            end = start + SEQUENCE_LENGTH
            valid_ratio = float(valid[start:end].mean())
            if valid_ratio >= MIN_VALID_FRAME_RATIO:
                yield poses[start:end].reshape(SEQUENCE_LENGTH, 51), int(frame_indices[start]), int(frame_indices[end - 1]), valid_ratio


counts = {s: 0 for s in ["train", "val", "test"]}
per_video_window_counts = {}
for i, row in enumerate(tracked_df.itertuples(index=False), 1):
    n_windows = sum(1 for _ in iter_video_windows(row.npz_path))
    counts[row.split] += n_windows
    per_video_window_counts[row.npz_path] = int(n_windows)
    if i % 500 == 0 or i == len(tracked_df):
        print(f"Counted {i:,}/{len(tracked_df):,} videos | train={counts['train']:,} val={counts['val']:,} test={counts['test']:,}")

print("\nSEQUENCE WINDOW COUNTS")
print(json.dumps(counts, indent=2))
if counts["train"] == 0 or counts["val"] == 0 or counts["test"] == 0:
    raise RuntimeError(f"At least one split has zero windows: {counts}. Do not train; inspect pose quality and source splits.")

Counted 500/13,319 videos | train=0 val=0 test=2,249
Counted 1,000/13,319 videos | train=0 val=0 test=4,521
Counted 1,500/13,319 videos | train=0 val=0 test=7,414
Counted 2,000/13,319 videos | train=0 val=0 test=11,198
Counted 2,500/13,319 videos | train=0 val=0 test=13,519
Counted 3,000/13,319 videos | train=2,383 val=0 test=14,452
Counted 3,500/13,319 videos | train=3,455 val=0 test=14,452
Counted 4,000/13,319 videos | train=6,144 val=0 test=14,452
Counted 4,500/13,319 videos | train=8,688 val=0 test=14,452
Counted 5,000/13,319 videos | train=10,537 val=0 test=14,452
Counted 5,500/13,319 videos | train=12,438 val=0 test=14,452
Counted 6,000/13,319 videos | train=13,906 val=0 test=14,452
Counted 6,500/13,319 videos | train=17,369 val=0 test=14,452
Counted 7,000/13,319 videos | train=19,114 val=0 test=14,452
Counted 7,500/13,319 videos | train=23,037 val=0 test=14,452
Counted 8,000/13,319 videos | train=29,513 val=0 test=14,452
Counted 8,500/13,319 videos | train=30,864 val=0 test=14,4

In [4]:
# CELL 4 — Create disk-backed split arrays and write windows incrementally
# X has the same format expected by Notebook 05: [number_of_windows, 16, 51].
memmaps = {}
offsets = {s: 0 for s in ["train", "val", "test"]}

for split in ["train", "val", "test"]:
    shape = (counts[split], SEQUENCE_LENGTH, 51)
    X = np.lib.format.open_memmap(OUT_DIR / f"X_{split}.npy", mode="w+", dtype=np.float32, shape=shape)
    y = np.lib.format.open_memmap(OUT_DIR / f"y_{split}.npy", mode="w+", dtype=np.int64, shape=(counts[split],))
    videos = np.lib.format.open_memmap(OUT_DIR / f"videos_{split}.npy", mode="w+", dtype=np.int32, shape=(counts[split],))
    memmaps[split] = {"X": X, "y": y, "videos": videos}

per_video_rows = []
for i, row in enumerate(tracked_df.itertuples(index=False), 1):
    before = offsets[row.split]
    class_id = label_to_id[row.label]
    for window, start_frame, end_frame, valid_ratio in iter_video_windows(row.npz_path):
        j = offsets[row.split]
        memmaps[row.split]["X"][j] = window
        memmaps[row.split]["y"][j] = class_id
        memmaps[row.split]["videos"][j] = int(row.video_id)
        offsets[row.split] += 1

    per_video_rows.append({
        "video_id": int(row.video_id),
        "video_path": row.video_path,
        "label": row.label,
        "split": row.split,
        "recording_group_key": row.recording_group_key,
        "source_npz": row.npz_path,
        "target_track_id": int(row.target_track_id),
        "sampled_frames": int(row.sampled_frames),
        "target_pose_observations": int(row.target_pose_observations),
        "complete_video": bool(row.complete_video),
        "inference_device": row.inference_device,
        "sequence_windows": offsets[row.split] - before,
    })

    if i % 250 == 0 or i == len(tracked_df):
        for split in memmaps:
            for arr in memmaps[split].values():
                arr.flush()
        print(f"Wrote {i:,}/{len(tracked_df):,} videos | train={offsets['train']:,} val={offsets['val']:,} test={offsets['test']:,}")

for split in memmaps:
    for arr in memmaps[split].values():
        arr.flush()
del memmaps
gc.collect()

assert offsets == counts, f"Write/count mismatch: expected {counts}, wrote {offsets}"
print("All sequence arrays written successfully:", offsets)

Wrote 250/13,319 videos | train=0 val=0 test=1,022
Wrote 500/13,319 videos | train=0 val=0 test=2,249
Wrote 750/13,319 videos | train=0 val=0 test=3,603
Wrote 1,000/13,319 videos | train=0 val=0 test=4,521
Wrote 1,250/13,319 videos | train=0 val=0 test=5,711
Wrote 1,500/13,319 videos | train=0 val=0 test=7,414
Wrote 1,750/13,319 videos | train=0 val=0 test=10,067
Wrote 2,000/13,319 videos | train=0 val=0 test=11,198
Wrote 2,250/13,319 videos | train=0 val=0 test=12,559
Wrote 2,500/13,319 videos | train=0 val=0 test=13,519
Wrote 2,750/13,319 videos | train=646 val=0 test=14,452
Wrote 3,000/13,319 videos | train=2,383 val=0 test=14,452
Wrote 3,250/13,319 videos | train=2,992 val=0 test=14,452
Wrote 3,500/13,319 videos | train=3,455 val=0 test=14,452
Wrote 3,750/13,319 videos | train=5,090 val=0 test=14,452
Wrote 4,000/13,319 videos | train=6,144 val=0 test=14,452
Wrote 4,250/13,319 videos | train=7,581 val=0 test=14,452
Wrote 4,500/13,319 videos | train=8,688 val=0 test=14,452
Wrote 4,75

In [5]:
# CELL 5 — Save metadata, per-class support, and update Notebook 05 pointer
summary_df = pd.DataFrame(per_video_rows)
summary_df.to_csv(OUT_DIR / "video_sequence_summary.csv", index=False)

# Copy the frozen source split verbatim; no video/group reassignment is performed.
split_df.to_csv(OUT_DIR / "video_split_manifest.csv", index=False)
(OUT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

class_rows = []
for cid_text, class_name in sorted(label_map.items(), key=lambda item: int(item[0])):
    cid = int(cid_text)
    item = {"class_id": cid, "label": class_name}
    for split in ["train", "val", "test"]:
        yy = np.load(OUT_DIR / f"y_{split}.npy", mmap_mode="r", allow_pickle=False)
        vv = np.load(OUT_DIR / f"videos_{split}.npy", mmap_mode="r", allow_pickle=False)
        mask = (yy == cid)
        item[f"{split}_windows"] = int(mask.sum())
        item[f"{split}_videos"] = int(len(np.unique(vv[mask])))
        del yy, vv, mask
    class_rows.append(item)

class_support = pd.DataFrame(class_rows)
class_support.to_csv(OUT_DIR / "class_support_by_split.csv", index=False)

# Ensure the output rows still respect the exact frozen recording-group assignments.
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ga = set(summary_df.loc[summary_df.split.eq(a), "recording_group_key"].astype(str))
    gb = set(summary_df.loc[summary_df.split.eq(b), "recording_group_key"].astype(str))
    overlap = ga & gb
    assert not overlap, f"Output recording-group overlap detected: {a}/{b}: {list(overlap)[:5]}"

classes_without_train = class_support.loc[class_support.train_windows.eq(0), "label"].tolist()
classes_without_val = class_support.loc[class_support.val_windows.eq(0), "label"].tolist()
classes_without_test = class_support.loc[class_support.test_windows.eq(0), "label"].tolist()

metadata = {
    "experiment": "FULL UCF101 tracked pose sequences from YOLO predict + greedy IoU association + ViTPose",
    "source_group_split_manifest": str(SPLIT_PATH),
    "tracked_keypoint_directory": str(TRACKED_DIR),
    "output_directory": str(OUT_DIR),
    "expected_source_videos": int(len(split_df)),
    "tracked_source_videos": int(len(tracked_df)),
    "complete_source_videos": int(len(tracked_df)),
    "missing_source_videos": 0,
    "unreadable_or_invalid_npz_files": 0,
    "source_video_coverage": 1.0,
    "is_full_dataset": True,
    "is_pilot_or_partial": False,
    "sequence_length": int(SEQUENCE_LENGTH),
    "sequence_stride": int(SEQUENCE_STRIDE),
    "minimum_valid_frame_ratio": float(MIN_VALID_FRAME_RATIO),
    "minimum_valid_joints": int(MIN_VALID_JOINTS),
    "features_per_frame": 51,
    "windows_by_split": {k: int(v) for k, v in counts.items()},
    "recording_group_split_reused_without_changes": True,
    "person_selection": "most pose-valid observations by greedy IoU track selected in Notebook 03; numeric track ID is metadata, not a model feature",
    "tracking_method": "greedy IoU association on YOLO predict detections; not ByteTrack",
    "classes_without_train_windows": classes_without_train,
    "classes_without_val_windows": classes_without_val,
    "classes_without_test_windows": classes_without_test,
    "training_format": "X arrays [N,16,51]; y class IDs; videos source video IDs",
    "note": "Notebook 05 derives velocity, acceleration and bone channels from the 51 base pose features during training.",
}
(OUT_DIR / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# This pointer is the interface Notebook 05 already reads. It is updated only after all arrays/reports are saved.
POINTER = NOTEBOOK2_DIR / "data" / "CURRENT_TRACKED_SEQUENCE_DIR.txt"
POINTER.parent.mkdir(parents=True, exist_ok=True)
POINTER.write_text(str(OUT_DIR), encoding="utf-8")

print("FULL TRACKED SEQUENCE BUILD COMPLETE")
print(json.dumps(metadata, indent=2))
print("\nClass support saved:", OUT_DIR / "class_support_by_split.csv")
print("Per-video summary saved:", OUT_DIR / "video_sequence_summary.csv")
print("Dataset metadata saved:", OUT_DIR / "dataset_metadata.json")
print("Notebook 05 pointer updated:", POINTER)

if classes_without_train or classes_without_val:
    print("\nWARNING: Some classes have zero train/validation windows. Notebook 05 will reject missing train/validation classes.")
if classes_without_test:
    print("\nWARNING: Some classes have zero test windows; report supported-class metrics and inspect these classes.")

FULL TRACKED SEQUENCE BUILD COMPLETE
{
  "experiment": "FULL UCF101 tracked pose sequences from YOLO predict + greedy IoU association + ViTPose",
  "source_group_split_manifest": "C:\\Users\\admin\\ViTPose_Action_Recognition\\data\\processed\\sequences_group_split_v1\\video_split_manifest.csv",
  "tracked_keypoint_directory": "C:\\Users\\admin\\ViTPose_Action_Recognition\\Notebook_2_Improved_v2\\data\\tracked_keypoints_iou_v3_full_v1",
  "output_directory": "C:\\Users\\admin\\ViTPose_Action_Recognition\\Notebook_2_Improved_v2\\data\\sequences_tracked_group_split_iou_v3_full_v1\\20261031_035536",
  "expected_source_videos": 13319,
  "tracked_source_videos": 13319,
  "complete_source_videos": 13319,
  "missing_source_videos": 0,
  "unreadable_or_invalid_npz_files": 0,
  "source_video_coverage": 1.0,
  "is_full_dataset": true,
  "is_pilot_or_partial": false,
  "sequence_length": 16,
  "sequence_stride": 4,
  "minimum_valid_frame_ratio": 0.8,
  "minimum_valid_joints": 8,
  "features_per_fr

## After this notebook finishes

1. Confirm `dataset_metadata.json` reports `is_full_dataset: true`, `source_video_coverage: 1.0`, and all three split window counts are non-zero.
2. Inspect `class_support_by_split.csv`, especially classes with zero validation or test windows. Do not hide or silently rebalance these classes.
3. Notebook 05 reads `data/CURRENT_TRACKED_SEQUENCE_DIR.txt`; this notebook updates that pointer only after all arrays and reports have been saved.
4. Notebook 05 expects the `X_*` arrays to have shape `[N, 16, 51]`. This notebook preserves that interface.
5. The dataset contains one primary tracked person's pose per video window. Although all-person observations remain in source NPZ files, this builder does not combine multiple people into one skeleton.
6. If this notebook stops while writing, its timestamped output folder may be incomplete. Rerun it to create a new timestamped build; previous extraction NPZs remain untouched.

**Do not run Notebook 05 if the full coverage audit in Cell 2 failed.**